
# 01 - Unity Catalog Volumes and Data Landing

This notebook sets up the full data infrastructure for the rideshare project:

1. Create an external location pointing to ADLS storage
2. Create a catalog and schemas for organising data
3. Create volumes for landing source files and storing processed outputs
4. Copy raw source files (CSV, Parquet, JSON, Avro, XML) into the landing volume

By the end, the following structure is ready:

```text
rideshare_dev (catalog)
├── landing (schema)
│   └── source_files (volume) → trip/, trip_time/, zone_lookup/, payment/, drivers/
└── processed (schema)
    └── output_files (volume) → empty, ready for KPI outputs
```

> Run all cells top-to-bottom. If something goes wrong, use
> **Notebook 99 - Rideshare Project Cleanup and Reset** to start over.


> #### 1. Create the rideshare project folder manually in the Azure Portal


### ADLS project directory

```text
container-dev-dbx/
└── rideshare/
```


> #### 2. Create a Unity Catalog External Location pointing to the new rideshare/ ADLS folder

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS el_rideshare_dev
    URL 'abfss://container-dev-dbx@sadevdbxeus2.dfs.core.windows.net/rideshare'
    WITH (STORAGE CREDENTIAL ac_dev_dbx_eus2)
    COMMENT 'External location for the rideshare development project'

In [0]:
%sql DESCRIBE EXTERNAL LOCATION el_rideshare_dev;


> #### 3. Test the external location
>
> 1. Open **Catalog Explorer** → **External Locations** → `el_rideshare_dev`
> 2. Click **Test connection** (top-right)
> 3. All checks should show green: Read, List, Write, Delete, Path Exists,
>    Hierarchical Namespace Enabled, **File Events Read**

<details>
<summary><strong>🔧 Troubleshooting: File Events Read Failed (click to expand)</strong></summary>

---

**Step 1 — Verify the four required Azure roles are assigned**

Go to Azure Portal → Storage Account (`sadevdbxeus2`) → Access Control (IAM).
Confirm the access connector's managed identity (`ac_dev_dbx_eus2`) has:

1. Storage Account Contributor
2. Storage Blob Data Contributor
3. EventGrid EventSubscription Contributor
4. Storage Queue Data Contributor

---

**Step 2 — Check for ABAC conditions on role assignments**

Look at the **Condition** column in the role assignments list.
If any role shows "Add" (instead of "None"), it has a restricting condition.

**Fix:** Delete the conditioned role assignment, then re-add the same role
**without** conditions (select "Not constrained" on the Conditions tab).

*Note: The conditions editor won't let you save with zero conditions —
you must delete and re-create the assignment.*

---

**Step 3 — Check storage account networking**

Go to Storage Account → Networking. Confirm:
- **Public network access** = "Enabled from all networks", OR
- If firewalled: "Allow Azure services on the trusted services list" is checked

The queue endpoint (`*.queue.core.windows.net`) must be reachable from
the Databricks control plane.

---

**Step 4 — Wait for role propagation and re-test**

Azure role changes can take **5–10 minutes** to propagate.
After fixing roles, wait a few minutes, then click **Test connection** again.
The UI shows the cached last result until you explicitly re-run it.

</details>


> #### 4. Create the `rideshare_dev` catalog with a dedicated managed storage path

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS rideshare_dev
MANAGED LOCATION
'abfss://container-dev-dbx@sadevdbxeus2.dfs.core.windows.net/rideshare/uc-managed'
COMMENT 'Catalog for the rideshare development project';

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
USE CATALOG rideshare_dev;

In [0]:
%sql
SELECT current_catalog();

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS rideshare_dev.landing
COMMENT 'Incoming rideshare source files';

In [0]:
%sql
SHOW SCHEMAS IN rideshare_dev;


> #### 5. Create an external volume for the landing area, then create five dataset folders inside it

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS rideshare_dev.landing.source_files
LOCATION 'abfss://container-dev-dbx@sadevdbxeus2.dfs.core.windows.net/rideshare/landing'
COMMENT 'Landing volume for original rideshare source files';

In [0]:
# Create one folder per dataset inside the landing volume

volume_path = "/Volumes/rideshare_dev/landing/source_files"

source_folders = [
    "trip",
    "trip_time",
    "zone_lookup",
    "payment",
    "drivers",
]

for folder in source_folders:
    dbutils.fs.mkdirs(f"{volume_path}/{folder}")

In [0]:
# Confirm the folders were created
display(dbutils.fs.ls(volume_path))

In [0]:
%sql
DESCRIBE VOLUME rideshare_dev.landing.source_files;


> #### 6. Copy source files from the Git repository to the landing volume

```text
Find the repository root
        ↓
Map each Git source file to its landing folder
        ↓
Create the destination folders and copy the files unchanged
```

In [0]:
import shutil
from pathlib import Path

# --- Step 1: Dynamically find the repository root ---
# Walk up from the current working directory until we find the data/raw folder.
# This makes the code portable across different users and workspaces.

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "raw").is_dir():
            return path
    raise FileNotFoundError("Could not find repository folder containing data/raw")

repo_root = find_repo_root(Path.cwd())

# --- Step 2: Define source-to-destination file mapping ---
# Source: Git repo (data/raw/...)  →  Destination: Volume landing folders

volume_root = Path("/Volumes/rideshare_dev/landing/source_files")

file_map = {
    "data/raw/csv/trip.csv":              "trip/trip.csv",
    "data/raw/parquet/trip_time.parquet": "trip_time/trip_time.parquet",
    "data/raw/json/zone_lookup.json":     "zone_lookup/zone_lookup.json",
    "data/raw/avro/payment.avro":         "payment/payment.avro",
    "data/raw/xml/drivers.xml":           "drivers/drivers.xml",
}

# --- Step 3: Copy each file (overwrites if already exists) ---

for src_rel, dst_rel in file_map.items():
    src = repo_root / src_rel
    dst = volume_root / dst_rel
    shutil.copy2(src, dst)
    print(f"✓ {src_rel} → {dst_rel}")

# --- Step 4: Verify all files landed correctly ---

print("\n--- Verification ---")
for dst_rel in file_map.values():
    dst = volume_root / dst_rel
    print(f"{dst_rel}: exists={dst.exists()}, size={dst.stat().st_size} bytes")


> #### 7. Create a separate processed schema and destination external volume

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS rideshare_dev.processed
COMMENT 'Processed file outputs for the rideshare project';

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS rideshare_dev.processed.output_files
LOCATION 'abfss://container-dev-dbx@sadevdbxeus2.dfs.core.windows.net/rideshare/processed'
COMMENT 'Destination volume for processed rideshare files';

In [0]:
%sql
DESCRIBE VOLUME rideshare_dev.processed.output_files;

In [0]:
# Confirm the processed volume exists and is empty (no outputs yet)
output_root = "/Volumes/rideshare_dev/processed/output_files"

display(dbutils.fs.ls(output_root))


---
### Setup complete

You now have:

| Object | Name | Purpose |
|--------|------|--------|
| External Location | `el_rideshare_dev` | Connects Databricks to the ADLS rideshare/ folder |
| Catalog | `rideshare_dev` | Top-level container for all rideshare data |
| Schema | `rideshare_dev.landing` | Holds raw source files as-is |
| Schema | `rideshare_dev.processed` | Will hold transformed outputs |
| Volume | `landing.source_files` | 5 datasets in original formats (CSV, Parquet, JSON, Avro, XML) |
| Volume | `processed.output_files` | Empty, ready for KPI outputs |

**Next notebook:** Read the source files using Spark and write processed outputs.